# Final Results

We collect here the models we have chosen for each borough thus far. Our goal here is to show the final model training, evaluation, and relevant plots. What is reproduced here can be found in the evaluations notebooks inside the folders for the boroughs and citywide level e.g. [5evaluations.ipynb](./citywide/5evaluations.ipynb) in the citywide folder.

<div style="text-align:center">

|         Level         |                                        Chosen Model                                                    |                  Folder                 |
| :-------------------: | :----------------------------------------------------------------------------------------------------: | :-------------------------------------: |
|        Citywide       |                        [Hybrid Prophet + XGBoost](./citywide/5evaluations.ipynb)                       |         [citywide](./citywide/)         |
|       Manhattan       |                         [Prophet](./manhattan/2neural_solo_prophet.ipynb)                        |        [manhattan](./manhattan/)        |
|        Brooklyn       |                              [NeuralProphet](./brooklyn/3evaluation.ipynb)                             |         [brooklyn](./brooklyn/)         |
|      Staten Island    |                         [Prophet](./staten_island/1modeling_experiments.ipynb)                         | [staten_island](./staten_island/)       |
|         Queens        | [Prophet](./bronx_and_queens/1modeling_experiments.ipynb) | [bronx_and_queens](./bronx_and_queens/)    |                                         |
|         Bronx         |                        [Prophet](./bronx_and_queens/1modeling_experiments.ipynb)                       | [bronx_and_queens](./bronx_and_queens/) |

</div>


*To improve readability and maintainability, it's better to refactor the code into functions, instead of relying on global variables or variables defined across multiple Jupyter notebook cells.

## Load Packages

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
import xgboost as xgb
import optuna
from neuralprophet import NeuralProphet
import logging
import requests


/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


## Time Series Split

In [2]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

## Brooklyn

In [3]:
np.NaN = np.nan

def load_best_params(study_name: str, storage_url: str):
    study = optuna.load_study(study_name=study_name, storage=storage_url)
    best_params = study.best_params
    best_value = study.best_value  # Optional: best objective value

    return best_params, best_value

study_name = "model_neural"
storage_url = "sqlite:///brooklyn/model_neural.db"

best_params, best_value = load_best_params(study_name, storage_url)

In [4]:
def load_rat_data(data_path: str, borough_name: str, start_date: str, end_date: str):
    # Load the rat sightings data
    rs = pd.read_csv(data_path)
    rs['created_date'] = pd.to_datetime(rs['created_date'])
    rs = rs[(rs['created_date'] >= start_date) & (rs['created_date'] < end_date)]

    # Restrict to the specified borough
    rs = rs[rs['borough'] == borough_name]

    # Drop the 'borough' column
    rs = rs.drop(columns=['borough'])

    # Rename columns for Prophet
    rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
    return rs

def load_weather_data():
    nd = pd.read_csv("../scr/data/weather_data/wd_2020_to_march_2026.csv")
    wd = nd
    wd = wd.set_index('time')
    wd = nd
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
    wd = wd[['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum', 'ds']]
    return wd

In [5]:
start_date = "2020-01-01"
end_date = "2026-02-28"

rs = load_rat_data('../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv', "BROOKLYN", start_date, end_date)

In [6]:
wd = load_weather_data()

In [7]:
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

In [8]:
wd["ds"] = pd.to_datetime(wd["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])

# Merge weather data with rat sightings data
rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

# Prepare lag values for regressed features
lags_for_regressed_features = {
    'apparent_temperature_max': best_params['lag_temp_max'],
    'apparent_temperature_min': best_params['lag_temp_min'],
    'snowfall_sum': best_params['lag_snowfall']}


In [ ]:
# Initialize list to store results
results = []

# Loop over the splits
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    print(f"Processing fold {i + 1}")
    train = rs.iloc[train_index].copy()
    train = train.dropna(subset=["y"])

    test = rs.iloc[test_index].copy()

    # Initialize NeuralProphet model
    model = NeuralProphet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        learning_rate=best_params['learning_rate'],
        epochs=best_params['epochs'],
        n_lags=best_params['n_lags'],
        ar_reg=best_params['ar_reg'],
        accelerator="auto",  # uses GPU if available
        batch_size=best_params['batch_size']
    )

    # Add US holidays
    model = model.add_country_holidays(country_name="US")

    # Add lagged regressors
    for column in regressed_features:
        model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])

    # Train the model
    print(f"Training the model for fold {i+1}.")

    model.fit(train progress="True")

    # Prepare future dataframe with regressors
    future = pd.concat([train[['ds', 'y'] + regressed_features], 
                        test[['ds', 'y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
    
    print(f"Making predictions from model for fold {i+1}.")
    forecast = model.predict(future)

    print(f"Rounding forecast for fold {i+1}.")
    # Evaluate the model
    y_pred = forecast["yhat1"].iloc[-len(test):].values
    y_pred = np.round(y_pred)
    y_true = test["y"].values


    print(f"Evaluating forecast for fold {i+1}.")
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    # Append the results for this fold
    print(f"Appending results for fold {i+1}.")
    results.append({"fold": i, "rmse": rmse, "mape": mape})

# Convert results to a DataFrame
neural_prophet_results_df = pd.DataFrame(results)
neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]


WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.947% of the data.
INFO - (NP.df_utils._infer_frequency) - Dataframe freq automatically defined as D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.


Processing fold 1
Training the model for fold 1.


: 

In [ ]:
brooklyn_results = neural_prophet_results_df.copy()

## Citywide

In [ ]:
# This cell defines functions to add features for XGBoost 

def create_features(df):
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

def add_cyclic(df):
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

def add_lags(df):
    # lags
    target_map = df['y'].to_dict()
    df['lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    return df

def add_seasonal_lags(df):
    # lags of various lengths for different levels of seasonality
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df

def add_moving_averages(df):
    df = df.copy()
    df = df.sort_index()
    # Must shift by 14 days because we do not want to let there be temporal leakage in our evaluations
    df['ma7'] = df['y'].shift(14).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(14).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(14).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(14).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(14).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(14).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(14).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(14).rolling(window=365).mean()
    
    return df


## Add weather data.
lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()


wd = pd.DataFrame(data["daily"])
wd["date"] = pd.to_datetime(wd["time"])
wd = wd.set_index("date")


def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    overlap = wd.columns.intersection(df.columns)
    wd = wd.drop(columns=overlap)
    df = df.join(wd, how="left")
    return df

def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['apparent_temperature_min_lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['apparent_temperature_min_lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df['apparent_temperature_min_lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)
    df['apparent_temperature_min_lag18'] = (df.index - pd.Timedelta('18 days')).map(target_map)
    df['apparent_temperature_min_lag19'] = (df.index - pd.Timedelta('19 days')).map(target_map)
    df['apparent_temperature_min_lag20'] = (df.index - pd.Timedelta('20 days')).map(target_map)
    df['apparent_temperature_min_lag21'] = (df.index - pd.Timedelta('21 days')).map(target_map)

    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    target_map = df['temperature_2m_max'].to_dict()
    df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

    return df


date_range = pd.date_range(start="2020-01-01", end="2026-02-28")
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())
federal_holidays = pd.DataFrame({'holiday': 'federal_us', 'ds': pd.to_datetime(holidays), 'lower_window': 0, 'upper_window': 1})

holidays = federal_holidays


def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())

    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format handling
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD format handling
                holidays = holidays.append(pd.to_datetime([d]))
                
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    start_dt = pd.to_datetime(start_date)
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df

def add_new_lags(df, x):
    # lags
    target_map = df[x].to_dict()
    df[f'{x}lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df[f'{x}lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)
    df[f'{x}lag18'] = (df.index - pd.Timedelta('18 days')).map(target_map)
    df[f'{x}lag19'] = (df.index - pd.Timedelta('19 days')).map(target_map)
    df[f'{x}lag20'] = (df.index - pd.Timedelta('20 days')).map(target_map)
    df[f'{x}lag21'] = (df.index - pd.Timedelta('21 days')).map(target_map)

    df[f'{x}lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df[f'{x}lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df[f'{x}lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    return df

In [ ]:
# These are all of the features considered

ALL_FEATURES = {'dayofweek', 'quarter', 'month', 'year', 'dayofyear',
       'dayofmonth', 'weekofyear', 'dayofweek_sin', 'dayofweek_cos',
       'month_sin', 'month_cos', 'lag15', 'lag16', 'lag30', 'lag60', 'lag90',
       'lag120', 'lag150', 'lag180', 'lag362', 'lag363', 'lag364', 'lag365',
       'lag366', 'lag367', 'lag730', 'lag1095', 'lag1460', 'lag1825', 'ma7',
       'ma30', 'ma60', 'ma90', 'ma120', 'ma150', 'ma180', 'ma365',
       'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
       'apparent_temperature_max', 'apparent_temperature_min',
       'apparent_temperature_mean', 'precipitation_sum', 'snowfall_sum',
       'apparent_temperature_min_lag14', 'apparent_temperature_min_lag15',
       'apparent_temperature_min_lag16', 'apparent_temperature_min_lag17',
       'apparent_temperature_min_lag18', 'apparent_temperature_min_lag19',
       'apparent_temperature_min_lag20', 'apparent_temperature_min_lag21',
       'apparent_temperature_min_lag30', 'apparent_temperature_min_lag60',
       'apparent_temperature_min_lag90', 'apparent_temperature_min_lag120',
       'apparent_temperature_min_lag150', 'apparent_temperature_min_lag180',
       'apparent_temperature_min_lag210', 'apparent_temperature_min_lag240',
       'apparent_temperature_min_lag270', 'apparent_temperature_min_lag300',
       'apparent_temperature_min_lag330', 'apparent_temperature_min_lag360',
       'apparent_temperature_min_lag365', 'apparent_temperature_min_lag730',
       'temperature_2m_max_lag14', 'temperature_2m_max_lag30',
       'temperature_2m_max_lag60', 'is_federal_holiday', 'Trash_Law',
       'New_Trash_Law', 'Rat_Mitigation_Zone', 'Rat_Czar_Appointed',
       'residuals', 'residualslag15', 'residualslag16', 'residualslag17',
       'residualslag18', 'residualslag19', 'residualslag20', 'residualslag21',
       'residualslag30', 'residualslag365', 'residualslag730', 'trend',
       'yhat_lower', 'yhat_upper'}

In [ ]:
def load_study(db_path, study_name, direction="minimize"):
    return optuna.create_study(
        direction=direction,
        study_name=study_name,
        storage=db_path,
        load_if_exists=True)

def extract_best_params(study):
    # Extracts the best parameters, features, hyperparameters from the study.
    best_params = study.best_params
    best_features = [f for f in ALL_FEATURES if best_params.get(f, False)]
    best_hyperparams = {k: v for k, v in best_params.items() if k not in ALL_FEATURES}

    return best_params, best_features, best_hyperparams

def extract_param_subset(hyperparams, keys):
    # return hyperparameters for use
    return {k: hyperparams[k] for k in keys if k in hyperparams}

study = load_study(db_path="sqlite:///citywide/xgbprophet_model26.db", study_name="hybrid_model_feature_parameter_search")

# Extract best parameters and hyperparameters
best_params, best_features, best_hyperparams = extract_best_params(study)

In [ ]:
print("Best Params:", best_params)
print("\nBest RMSE:", study.best_value)
print("\nBest Features:", best_features)

# Extract prophet and xgb hyperparameters
prophet_keys = ["changepoint_prior_scale", "seasonality_prior_scale", "holidays_prior_scale"]
best_prophet_params = extract_param_subset(best_hyperparams, prophet_keys)
xgb_keys = ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'colsample_bytree', 'gamma', 'min_child_weight', 'reg_lambda', 'reg_alpha']
best_xgb_params = extract_param_subset(best_hyperparams, xgb_keys)

print("\nBest Prophet Parameters:")
print(best_prophet_params)

print("\nBest XGBoost Parameters:")
print(best_xgb_params)

In [ ]:
# # Load in data
# rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
# rs['created_date'] = pd.to_datetime(rs['created_date']) 
# rs = rs[rs['created_date']<'2026-03-01']
# rs = rs[rs['created_date']>='2020-01-01']
# rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# save = rs['ds'].copy().values
# rs = rs.set_index('ds')
# rs.index = pd.to_datetime(rs.index)
# rs['ds']=save
# rs = create_features(rs)
# rs = add_cyclic(rs)
# rs = add_lags(rs)
# rs = add_seasonal_lags(rs)
# rs = add_moving_averages(rs)
# rs = add_weather_data(rs,wd)
# rs = add_more_weather_feature(rs)
# rs = add_federal_holidays(rs, custom_holidays = ['12-31'])
# rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
# rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
# rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
# rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')
# FEATURES = best_features

In [ ]:
def load_and_preprocess_data(file_path, weather_data):
    # Load in data
    rs = pd.read_csv(file_path)
    rs['created_date'] = pd.to_datetime(rs['created_date'])

    # Filter data within the desired date range
    rs = rs[(rs['created_date'] >= '2020-01-01') & (rs['created_date'] < '2026-03-01')]

    # Group by date and count the occurrences
    rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
    rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

    # Save original dates
    save = rs['ds'].copy().values
    rs = rs.set_index('ds')
    rs.index = pd.to_datetime(rs.index)
    rs['ds'] = save

    # Feature engineering
    rs = create_features(rs)
    rs = add_cyclic(rs)
    rs = add_lags(rs)
    rs = add_seasonal_lags(rs)
    rs = add_moving_averages(rs)
    rs = add_weather_data(rs, weather_data)
    rs = add_more_weather_feature(rs)
    rs = add_federal_holidays(rs, custom_holidays=['12-31'])
    
    # Add law flags with specific start dates
    rs = add_law_flag(rs, law_name='Trash_Law', start_date='2024-03-01')
    rs = add_law_flag(rs, law_name='New_Trash_Law', start_date='2024-11-01')
    rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date='2023-07-07')
    rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date='2023-04-12')

    return rs

In [ ]:
rs = load_and_preprocess_data('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv', wd)

In [ ]:
FEATURES = best_features  # best_features already a global variable

In [ ]:
def forward_fill_rows(df, seed_row, include_columns=None):
    df_ffilled = df.copy()
    
    if include_columns is None:
        # if no columns specified, forward-fill all
        include_columns = df_ffilled.columns.tolist()
    
    for i in range(len(df_ffilled)):
        if i == 0:
            # first row takes seed_row for included columns
            for col in include_columns:
                df_ffilled.at[df_ffilled.index[i], col] = seed_row[col]
        else:
            # subsequent rows take previous row for included columns
            for col in include_columns:
                df_ffilled.at[df_ffilled.index[i], col] = df_ffilled.at[df_ffilled.index[i-1], col]
                
    return df_ffilled

In [ ]:
def train_and_evaluate(tscv, rs, best_prophet_params, best_xgb_params, holidays, FEATURES):
    results = []

    for i, (train_index, test_index) in enumerate(tscv.split(rs)):
        train = rs.iloc[train_index].copy()
        test = rs.iloc[test_index].copy()

        # Forward-fill test using the last row of train
        test = forward_fill_rows(test, 
                                seed_row=train.iloc[-1], 
                                include_columns=['temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
                                                 'apparent_temperature_max', 'apparent_temperature_min', 
                                                 'apparent_temperature_mean', 'precipitation_sum', 'snowfall_sum'])

        # Train Prophet model
        model = Prophet(**best_prophet_params, holidays=holidays)
        model.add_country_holidays(country_name='US')
        model.fit(train)

        # Generate Prophet predictions on the training set
        train_future = model.make_future_dataframe(periods=0, freq='D')
        train_forecast = model.predict(train_future)

        # Calculate residuals
        train_residuals = train['y'].values - train_forecast['yhat'].values
        residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals})
        train['residuals'] = train_residuals

        # Add new lags and other Prophet outputs
        add_new_lags(train, 'residuals')
        train['trend'] = train_forecast['trend'].values
        train['yhat_lower'] = train_forecast['yhat_lower'].values
        train['yhat_upper'] = train_forecast['yhat_upper'].values

        # Prepare the data for XGBoost
        X_train_residuals = train[FEATURES]
        y_train_residuals = residuals_df['y']

        # Train XGBoost on residuals
        xgb_model = xgb.XGBRegressor(**best_xgb_params)
        xgb_model.fit(X_train_residuals, y_train_residuals)

        # Prepare test data
        test['residuals'] = np.nan  # Initialize residuals for the test set
        dummy = pd.concat([train, test], axis=0)
        add_new_lags(dummy, 'residuals')  # Add lags for the test set

        test = dummy.iloc[test_index].copy()  # Restore test set

        # Prophet forecast for the test set
        future = model.make_future_dataframe(periods=len(test), freq='D')
        prophet_forecast = model.predict(future)

        # Add Prophet forecast features to the test set
        test.loc[:, 'trend'] = prophet_forecast[-len(test):]['trend'].values
        test.loc[:, 'yhat_lower'] = prophet_forecast[-len(test):]['yhat_lower'].values
        test.loc[:, 'yhat_upper'] = prophet_forecast[-len(test):]['yhat_upper'].values

        # XGBoost predictions on residuals
        X_test = test[FEATURES]
        xgb_residual_preds = xgb_model.predict(X_test)

        # Combine Prophet and XGBoost predictions
        y_pred = np.round(prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds)
        y_true = test['y'].values

        # Calculate RMSE for this fold
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        results.append(rmse)  # Store the RMSE for this fold

    # Convert the results into a DataFrame
    prophet_xgb_results_df = pd.DataFrame(results, columns=["RMSE"])

    # Calculate the mean RMSE across all folds
    mean_rmse = prophet_xgb_results_df['RMSE'].mean()
    prophet_xgb_results_df.loc['mean'] = [mean_rmse]

    return prophet_xgb_results_df

In [ ]:
citywide_results = train_and_evaluate(tscv, rs, best_prophet_params, best_xgb_params, holidays, FEATURES)

## Manhattan

In [ ]:
def run_prophet_time_series_split(borough_name: str, data_path: str, tscv) -> pd.DataFrame:
    # Read data
    rs = pd.read_csv(data_path)
    rs['created_date'] = pd.to_datetime(rs['created_date']) 

    # Filter data between 2020-01-01 and 2026-02-28
    rs = rs[(rs['created_date'] >= '2020-01-01') & (rs['created_date'] < '2026-03-01')]

    # Restrict to the specified borough
    rs = rs[rs['borough'] == borough_name]

    # Drop the column with borough
    rs = rs.drop(columns=['borough'])

    # Create a continuous date index with zero-filled values for missing dates
    rs['created_date'] = pd.to_datetime(rs['created_date'])
    start = rs['created_date'].min()
    end = rs['created_date'].max()
    rs = (rs.set_index('created_date')
          .reindex(pd.date_range(start, end, freq='D'), fill_value=0)
          .rename_axis('created_date')
          .reset_index())

    # Generate US federal holidays
    date_range = pd.date_range(start="2020-01-01", end="2026-02-28")
    calendar = USFederalHolidayCalendar()
    holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

    # Build the holidays DataFrame in the same structure as your original
    federal_holidays = pd.DataFrame({
        'holiday': 'federal_us',
        'ds': pd.to_datetime(holidays),
        'lower_window': 0,
        'upper_window': 1,
    })

    # Rename columns for Prophet model
    rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

    # Initialize TimeSeriesSplit
    results = []

    for i, (train_index, test_index) in enumerate(tscv.split(rs)):
        train = rs.iloc[train_index]
        test = rs.iloc[test_index]

        # Initialize and train Prophet model
        model = Prophet(holidays=federal_holidays)
        model.add_country_holidays(country_name='US')
        model.fit(train)

        # Make future predictions
        future = model.make_future_dataframe(periods=len(test), freq='D')
        forecast = model.predict(future)

        # Obtain predicted values and compare against the actuals
        y_pred = forecast['yhat'][-len(test):].values
        y_true = test['y'].values
        y_pred = np.round(y_pred)

        # Calculate RMSE
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))

        # Calculate MAPE
        mape = mean_absolute_percentage_error(y_true, y_pred)

        # Append results for this fold
        results.append({'fold': i, 'rmse': rmse, 'mape': mape})

    # Convert results to a DataFrame
    prophet_results_df = pd.DataFrame(results)
    prophet_results_df.loc['mean'] = ['mean', prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]

    return prophet_results_df

In [ ]:
data_path = '../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv'
manhattan_results = run_prophet_time_series_split("MANHATTAN", data_path, tscv)

## Staten Island & Bronx & Queens

In [ ]:
data_path = '../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv'
staten_island_results = run_prophet_time_series_split("STATEN ISLAND", data_path, tscv)
bronx_results = run_prophet_time_series_split("BRONX", data_path, tscv)
queens_results = run_prophet_time_series_split("QUEENS", data_path, tscv)

## Key Plots and Tables

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

results = {"Citywide": citywide_results, 
           "Manhattan": manhattan_results, 
           "Brooklyn": brooklyn_results,
           "Staten Island": staten_island_results, 
           "Bronx": bronx_results,
           "Queens": queens_results}

plt.figure(figsize=(12, 6))


# The shapes of the dataframe for citywide_results and {borough}_results differ so this code is clunkier than usual

for name, df in results.items():
    df_plot = df.copy()

    if 'rmse' in df_plot.columns:
        # Citywide: extract mean RMSE from 'mean' row
        if 'mean' in df_plot['fold'].values:
            mean_rmse = float(df_plot.loc[df_plot['fold'] == 'mean', 'rmse'])
            df_plot = df_plot[df_plot['fold'] != 'mean']  # drop mean row for plotting
        else:
            mean_rmse = df_plot['rmse'].mean()

        folds = df_plot['fold'].astype(str)
        rmse_values = df_plot['rmse']

    else:
        # Boroughs: mean row is an index
        if 'mean' in df_plot.index:
            mean_rmse = float(pd.to_numeric(df_plot.loc['mean'].values[0], errors='coerce'))
            df_plot = df_plot.loc[df_plot.index != 'mean']
        else:
            rmse_col = df_plot.columns[0]
            mean_rmse = pd.to_numeric(df_plot[rmse_col], errors='coerce').mean()

        # Convert RMSE to numeric, drop any NaNs
        rmse_col = df_plot.columns[0]
        df_plot[rmse_col] = pd.to_numeric(df_plot[rmse_col], errors='coerce')
        df_plot = df_plot.dropna()

        folds = df_plot.index.astype(str)
        rmse_values = df_plot[rmse_col]

    plt.plot(folds, rmse_values, marker='o', label=f"{name} (mean RMSE={mean_rmse:.2f})")

plt.xlabel("Fold")
plt.ylabel("RMSE")
plt.title("RMSE per Fold for Final Models -- Citywide and Borough")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

rmse_data = {}

for name, df in results.items():
    df_plot = df.copy()
    
    if 'rmse' in df_plot.columns:
        df_plot = df_plot[df_plot['fold'] != 'mean']
        series = df_plot.set_index('fold')['rmse'].astype(float)
        series.index = series.index.astype(str)
    else:
        df_plot = df_plot.loc[df_plot.index != 'mean']
        rmse_col = df_plot.columns[0]
        series = pd.to_numeric(df_plot[rmse_col], errors='coerce').dropna()
        series.index = series.index.astype(str)

    rmse_data[name] = series

rmse_df = pd.DataFrame(rmse_data)
mean_row = rmse_df.mean(axis=0)
mean_row.name = 'mean'
rmse_df = pd.concat([rmse_df, mean_row.to_frame().T])
rmse_df = rmse_df.reset_index().rename(columns={'index': 'fold'})


rmse_df